# THGS Arm-D Pipeline Diagnostic

D 의 인스턴스 분리 실패 원인을 **파이프라인 단계별** 로 시각·정량 추적한다.

```
입력 이미지
   ↓ ① FastSAM 4-conf       → raw mask 4세트
   ↓ ② mask NMS              → 정리된 mask 묶음
   ↓ ③ bbox crop → pad → 320 → ConvNeXt-L 입력 이미지
   ↓ ④ ConvNeXt-L (OpenCLIP) → 768-D feature
   ↓ ⑤ L2 정규화
   ↓ ⑥ 클러스터링 (cosine)   → 인스턴스 그룹
```

각 단계마다 **무엇이 문제인지** 시각으로 분리해서 본다.

---
## Cell 0. 환경 셋업


In [ ]:
# GPU + Drive + repo
!nvidia-smi -L
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE = "/content/drive/MyDrive/THGS_cache"
os.makedirs(DRIVE_CACHE, exist_ok=True)

REPO_URL = "https://github.com/BAEJUNHAK/THGS.git"
WORK_DIR = "/content/THGS"
if not os.path.exists(WORK_DIR):
    !git clone --recursive {REPO_URL} {WORK_DIR}
os.chdir(WORK_DIR)
!git pull


In [ ]:
# 의존성 (Phase 1 노트북에서 이미 깐 거면 빠르게 끝남)
!pip install -q --upgrade opencv-python
!pip install -q plyfile==0.8.1 omegaconf open_clip_torch transformers \
    pycocotools scikit-image scikit-learn einops natsort gdown \
    seaborn matplotlib tqdm pandas
!pip install -q ultralytics

import open_clip, ultralytics
print("open_clip:", open_clip.__version__, "| ultralytics:", ultralytics.__version__)


---
## Cell 1. 데이터·sample frame 로드 + helper


In [ ]:
import os, json, sys, copy, shutil, zipfile
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

# THGS 모듈 경로
sys.path.insert(0, '/content/THGS')
sys.path.insert(0, '/content/THGS/scripts')

# Re-use 인코더·평가기 함수
from encode_maskadapter_fastsam import (
    ConvNeXtCLIPVisual, mask2feat_convnext,
    CONVNEXT_INPUT, CONVNEXT_DIM,
)
from encode_fastsam_clip import (
    fastsam_4scale, masks_update, LEVEL_NAMES, CONF_LEVELS,
    pad_img, get_seg_img,
)
from eval_cluster_quality import (
    load_gt_masks_per_category, assign_gt_labels,
    cluster_purity, hungarian_mask_miou,
)
from ultralytics import FastSAM

DATASET, SCENE = 'lerf', 'ramen'
cfg = OmegaConf.load(f"configs/{DATASET}.yml")
DATA_PATH = cfg.dataset.data_path

SAMPLE_FRAME = "frame_00006"
IMG_PATH = f"{DATA_PATH}/{SCENE}/images/{SAMPLE_FRAME}.jpg"
GT_DIR = f"{DATA_PATH}/label/{SCENE}"
OUT_ROOT = "output/D_diagnostic"
os.makedirs(OUT_ROOT, exist_ok=True)

# === 1. LERF-OVS 데이터 (없으면 다운로드) ===
if not os.path.exists(IMG_PATH):
    print("LERF-OVS 데이터 없음 — 다운로드 시도")
    # 메인 노트북의 Drive 캐시에서 우선 복사
    drive_data = "/content/drive/MyDrive/THGS_cache/lerf_ovs"
    if os.path.exists(drive_data):
        print(f"  Drive 캐시에서 복사: {drive_data}")
        if not os.path.exists("data"):
            os.makedirs("data", exist_ok=True)
        if not os.path.exists("data/lerf_ovs"):
            shutil.copytree(drive_data, "data/lerf_ovs")
        if not os.path.exists(DATA_PATH) and os.path.exists("data/lerf_ovs"):
            os.symlink("lerf_ovs", DATA_PATH)
    else:
        # gdown 으로 다운로드
        import gdown
        LERF_DATA_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"
        LERF_DATA_ZIP = "/content/lerf_ovs.zip"
        gdown.download(id=LERF_DATA_ID, output=LERF_DATA_ZIP, quiet=False)
        with zipfile.ZipFile(LERF_DATA_ZIP, 'r') as z:
            z.extractall("data/")
        os.remove(LERF_DATA_ZIP)
        if not os.path.exists(DATA_PATH) and os.path.exists("data/lerf_ovs"):
            os.symlink("lerf_ovs", DATA_PATH)

assert os.path.exists(IMG_PATH), f"이미지 없음: {IMG_PATH}"
print(f"이미지 OK: {IMG_PATH}")

# === 2. FastSAM ckpt (없으면 다운로드) ===
FASTSAM_CKPT = "ckpts/FastSAM-x.pt"
os.makedirs("ckpts", exist_ok=True)
if not os.path.exists(FASTSAM_CKPT):
    print("FastSAM ckpt 없음 — 다운로드 시도")
    # 1차: ultralytics 자동 다운로드
    try:
        print("  ultralytics 자동 다운로드 시도...")
        m = FastSAM('FastSAM-x.pt')
        for c in ['FastSAM-x.pt',
                  getattr(m, 'ckpt_path', None),
                  os.path.expanduser('~/.config/Ultralytics/FastSAM-x.pt')]:
            if c and os.path.exists(c):
                shutil.copy2(c, FASTSAM_CKPT)
                print(f"    복사: {c} -> {FASTSAM_CKPT}")
                break
        del m
    except Exception as e:
        print(f"  ultralytics 다운로드 실패: {e}")

if not os.path.exists(FASTSAM_CKPT):
    print("  fallback: GitHub assets 직접 다운로드")
    !wget -q --show-progress -O {FASTSAM_CKPT} \
        https://github.com/ultralytics/assets/releases/download/v8.2.0/FastSAM-x.pt \
        || wget -q --show-progress -O {FASTSAM_CKPT} \
        https://github.com/CASIA-IVA-Lab/FastSAM/releases/download/v0.0.1/FastSAM-x.pt

assert os.path.exists(FASTSAM_CKPT) and os.path.getsize(FASTSAM_CKPT) > 10_000_000, \
    f"FastSAM 가중치 다운로드 실패: {FASTSAM_CKPT}"
print(f"FastSAM OK: {FASTSAM_CKPT} ({os.path.getsize(FASTSAM_CKPT)/1024/1024:.1f} MB)")

# === 3. 이미지 + GT 로드 ===
image_bgr = cv2.imread(IMG_PATH)
H, W = image_bgr.shape[:2]
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
print(f"\nframe={SAMPLE_FRAME}  HxW={H}x{W}")

gt_masks, _ = load_gt_masks_per_category(GT_DIR, SAMPLE_FRAME, target_hw=(H, W))
gt_categories = list(gt_masks.keys())
print(f"GT categories ({len(gt_categories)}): {gt_categories}")

plt.figure(figsize=(10, 7))
plt.imshow(image_rgb)
plt.title(f"{SAMPLE_FRAME}  ({W}x{H}, {len(gt_categories)} GT categories)")
plt.axis('off'); plt.show()


---
## Cell 2. Stage ① — FastSAM 4-conf raw mask

각 conf threshold 가 **다른 입자 크기의 mask** 를 만든다. 객체가 그어졌는지, 노이즈 mask 가 많은지 확인.


In [ ]:
# FastSAM 4-conf 실행 (raw mask 보기)
fastsam = FastSAM(FASTSAM_CKPT)
masks_per_level = fastsam_4scale(fastsam, image_bgr)

# 통계
for name, conf in zip(LEVEL_NAMES, CONF_LEVELS):
    print(f"  conf={conf} ({name:7s}) → {len(masks_per_level[name])} masks")

# 시각화: 2x2 grid, 각 level 의 mask 를 색깔별로 오버레이
def overlay_masks(image_rgb, masks_list, alpha=0.55, seed=0):
    out = image_rgb.copy().astype(np.float32) / 255.0
    rng = np.random.RandomState(seed)
    for m in masks_list:
        seg = m['segmentation']
        color = rng.rand(3)
        out[seg] = (1 - alpha) * out[seg] + alpha * color
    return np.clip(out, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name, conf in zip(axes.flat, LEVEL_NAMES, CONF_LEVELS):
    ax.imshow(overlay_masks(image_rgb, masks_per_level[name]))
    ax.set_title(f"conf={conf}  ({name})  N={len(masks_per_level[name])}")
    ax.axis('off')
plt.suptitle("Stage ① — FastSAM 4-conf raw masks", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/stage1_fastsam_4conf.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 3. Stage ② — mask NMS 정리

원본 노트북과 동일한 `iou_thr=0.8, score_thr=0.7, inner_thr=0.5` NMS 적용.
어떤 mask 가 살아남고 어떤 게 묻혔는지 확인.


In [ ]:
# NMS 전 통계
n_before = {n: len(masks_per_level[n]) for n in LEVEL_NAMES}
total_before = sum(n_before.values())

# NMS 적용
masks_d, masks_s, masks_m, masks_l = (masks_per_level[k] for k in LEVEL_NAMES)
masks_d, masks_s, masks_m, masks_l = masks_update(
    masks_d, masks_s, masks_m, masks_l,
    iou_thr=0.8, score_thr=0.7, inner_thr=0.5,
)
masks_after_nms = {'default': masks_d, 's': masks_s, 'm': masks_m, 'l': masks_l}
n_after = {k: len(v) for k, v in masks_after_nms.items()}
total_after = sum(n_after.values())

print(f"Before NMS: {n_before}  total={total_before}")
print(f"After  NMS: {n_after}  total={total_after}")
print(f"제거된 mask 개수: {total_before - total_after}")

# 시각화: NMS 전 (모든 4 level 합집합) vs NMS 후 (4 level 합집합)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
all_before = sum([masks_per_level[n] for n in LEVEL_NAMES], [])
all_after = sum([masks_after_nms[n] for n in LEVEL_NAMES], [])
axes[0].imshow(overlay_masks(image_rgb, all_before, seed=1))
axes[0].set_title(f"Before NMS  (N={len(all_before)})")
axes[0].axis('off')
axes[1].imshow(overlay_masks(image_rgb, all_after, seed=1))
axes[1].set_title(f"After  NMS  (N={len(all_after)})")
axes[1].axis('off')
plt.suptitle("Stage ② — NMS 전후", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/stage2_nms.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 4. Stage ③ — bbox crop → pad → 320 (ConvNeXt-L 가 실제로 보는 것)

각 mask 의 입력이 어떻게 변형되는지 직접 본다.
**핵심 의심**: 작은 mask 일수록 검정 영역이 320 의 큰 부분을 차지 → ConvNeXt-L 가 검정+객체 혼합으로 인코딩.


In [ ]:
# 모든 NMS-survived mask 를 평탄화
all_masks = []
for name in LEVEL_NAMES:
    for m in masks_after_nms[name]:
        all_masks.append({'level': name, **m})
print(f"진단 대상 mask 총 {len(all_masks)} 개")

# GT category 라벨 부여
# pseudo seg_map for assign_gt_labels: 가짜 (H,W) where each mask has consecutive id
fake_segmap = -np.ones((H, W), dtype=np.int64)
for i, m in enumerate(all_masks):
    fake_segmap[m['segmentation']] = i
mask_ids, gt_labels, best_ious = assign_gt_labels(
    fake_segmap, gt_masks, min_iou=0.2)
mask_id_to_label = {mid: lbl for mid, lbl in zip(mask_ids, gt_labels)}
mask_id_to_iou = {mid: iou for mid, iou in zip(mask_ids, best_ious)}

# 카테고리별 1개씩 대표 mask 선택 (best-IoU)
representatives = {}
for mid in mask_ids:
    lbl = mask_id_to_label[mid]
    if lbl == 'background':
        continue
    if lbl not in representatives or mask_id_to_iou[mid] > mask_id_to_iou[representatives[lbl]]:
        representatives[lbl] = mid

selected_ids = sorted(representatives.values(), key=lambda i: mask_id_to_label[i])
print(f"GT category 별 대표 mask: {len(selected_ids)} 개")
for mid in selected_ids:
    print(f"  mask_{mid:3d} → '{mask_id_to_label[mid]}' (best_IoU={mask_id_to_iou[mid]:.3f})")

# 시각화: K 행 × 4 열
K = len(selected_ids)
fig, axes = plt.subplots(K, 4, figsize=(16, 3 * K))
if K == 1:
    axes = axes.reshape(1, -1)
for r, mid in enumerate(selected_ids):
    m = all_masks[mid]
    label = mask_id_to_label[mid]
    iou = mask_id_to_iou[mid]

    # col 0: original with this mask highlighted
    overlay = image_rgb.copy().astype(np.float32) / 255.0
    overlay[m['segmentation']] = (
        0.4 * overlay[m['segmentation']] + 0.6 * np.array([1.0, 0.2, 0.2]))
    axes[r, 0].imshow(np.clip(overlay, 0, 1))
    axes[r, 0].set_title(f"[{label}]  mask_{mid}\nIoU(GT)={iou:.2f}")
    axes[r, 0].axis('off')

    # col 1: bbox crop with mask area only
    seg_img = get_seg_img(m, image_bgr)        # BGR HxWx3
    if seg_img.size == 0:
        axes[r, 1].text(0.5, 0.5, '(empty)', ha='center'); axes[r, 1].axis('off')
    else:
        axes[r, 1].imshow(cv2.cvtColor(seg_img, cv2.COLOR_BGR2RGB))
        axes[r, 1].set_title(f"bbox crop\n{seg_img.shape[1]}x{seg_img.shape[0]}")
        axes[r, 1].axis('off')

        # col 2: square padded
        padded = pad_img(seg_img)
        axes[r, 2].imshow(cv2.cvtColor(padded, cv2.COLOR_BGR2RGB))
        axes[r, 2].set_title(f"square padded\n{padded.shape[1]}x{padded.shape[0]}")
        axes[r, 2].axis('off')

        # col 3: resized to 320×320
        resized = cv2.resize(padded, (CONVNEXT_INPUT, CONVNEXT_INPUT))
        # fill ratio = 객체 픽셀 / 320^2
        fill_ratio = (resized.sum(axis=2) > 0).mean()
        axes[r, 3].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
        axes[r, 3].set_title(f"320 입력 (ConvNeXt-L)\nfill={fill_ratio*100:.1f}%")
        axes[r, 3].axis('off')

plt.suptitle("Stage ③ — ConvNeXt-L 가 실제로 보는 입력 (mask 별)",
             fontsize=14, y=1.001)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/stage3_crop_chain.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 5. Stage ④ — ConvNeXt-L 인코딩 + cosine 유사도 매트릭스

모든 NMS-survived mask 를 ConvNeXt-L 로 인코딩 → 768-D feature.
**같은 GT category 끼리 빨갛게**, 다른 카테고리끼리는 파랗게 나와야 정상.
다른 카테고리끼리 빨가면 → ConvNeXt-L 이 혼동 (D 의 핵심 실패 지점).


In [ ]:
# ConvNeXt-L 로 인코딩 (B-arm encoder 의 함수 그대로 재사용)
encoder = ConvNeXtCLIPVisual()

# 4 level 묶어서 한꺼번에 인코딩
feats_per_level = {}
seg_per_level = {}
for name in LEVEL_NAMES:
    masks = masks_after_nms[name]
    if len(masks) == 0:
        feats_per_level[name] = torch.zeros((0, CONVNEXT_DIM), dtype=torch.float16)
        continue
    emb, sm = mask2feat_convnext(masks, image_bgr, encoder)
    feats_per_level[name] = emb

# 평탄화: all_masks 순서와 같게 정렬
feat_list = []
for name in LEVEL_NAMES:
    feat_list.append(feats_per_level[name].float().numpy())
feat_all = np.concatenate(feat_list, axis=0)
print(f"feat shape: {feat_all.shape}  (mask {len(all_masks)} × dim {CONVNEXT_DIM})")
assert feat_all.shape[0] == len(all_masks)

# Cosine 유사도 매트릭스
feat_norm = feat_all / (np.linalg.norm(feat_all, axis=1, keepdims=True) + 1e-9)
cosine_M = feat_norm @ feat_norm.T

# GT category 순으로 정렬
order = sorted(range(len(all_masks)),
               key=lambda i: (mask_id_to_label.get(i, 'zzz'), -mask_id_to_iou.get(i, 0)))
sorted_labels = [mask_id_to_label.get(i, 'background') for i in order]
cosine_sorted = cosine_M[np.ix_(order, order)]

# 카테고리 경계
boundaries = []
prev = None
for k, lbl in enumerate(sorted_labels):
    if lbl != prev:
        boundaries.append((k, lbl))
        prev = lbl

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cosine_sorted, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='cosine')
# 카테고리 경계선
for k, lbl in boundaries:
    ax.axvline(k - 0.5, color='black', lw=0.4)
    ax.axhline(k - 0.5, color='black', lw=0.4)
# 축 레이블 (카테고리 별 중간 위치에 글자)
ticks = []
ticklabels = []
for i, (k, lbl) in enumerate(boundaries):
    end = boundaries[i+1][0] if i + 1 < len(boundaries) else len(sorted_labels)
    ticks.append((k + end) / 2)
    ticklabels.append(lbl)
ax.set_xticks(ticks); ax.set_xticklabels(ticklabels, rotation=60, ha='right', fontsize=8)
ax.set_yticks(ticks); ax.set_yticklabels(ticklabels, fontsize=8)
ax.set_title('Stage ④ — Mask-feature cosine matrix (GT category 순 정렬)\n'
             '같은 카테고리 블록 = 빨강이어야 정상', fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/stage4_cosine_matrix.png", dpi=120, bbox_inches='tight')
plt.show()

# 정량 진단: same-category pair vs diff-category pair 평균 유사도
labeled_idx = [i for i, l in enumerate(sorted_labels) if l != 'background']
same_pairs, diff_pairs = [], []
for i in labeled_idx:
    for j in labeled_idx:
        if i >= j: continue
        if sorted_labels[i] == sorted_labels[j]:
            same_pairs.append(cosine_sorted[i, j])
        else:
            diff_pairs.append(cosine_sorted[i, j])
print(f"\n=== 정량 진단 ===")
print(f"  same-category pair  cosine 평균 = {np.mean(same_pairs):.3f} (높을수록 좋음)")
print(f"  diff-category pair  cosine 평균 = {np.mean(diff_pairs):.3f} (낮을수록 좋음)")
print(f"  분리 margin                      = {np.mean(same_pairs) - np.mean(diff_pairs):+.3f}")
print(f"  → margin 이 0 에 가까우면 ConvNeXt 가 카테고리 간 의미 분리를 못함")


---
## Cell 6. Stage ⑤ — t-SNE 2D 산점도 (GT category 색칠)

mask feature 를 2D 로 투영해서 같은 카테고리끼리 뭉치는지 직접 본다.
같은 색깔이 모여있어야 정상 — 흩뿌려져 있으면 ConvNeXt-L feature 가 의미 분리 못함.


In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# 라벨 있는 mask 만 사용
labeled_mask_ids = [i for i in range(len(all_masks))
                    if mask_id_to_label.get(i, 'background') != 'background']
sub_feat = feat_norm[labeled_mask_ids]
sub_lbl = [mask_id_to_label[i] for i in labeled_mask_ids]
print(f"라벨 있는 mask: {len(labeled_mask_ids)} / {len(all_masks)}")

if len(sub_feat) >= 5:
    tsne = TSNE(n_components=2, perplexity=min(8, len(sub_feat) - 1),
                init='pca', random_state=0)
    xy_tsne = tsne.fit_transform(sub_feat)
else:
    xy_tsne = PCA(n_components=2).fit_transform(sub_feat)
xy_pca = PCA(n_components=2).fit_transform(sub_feat)

cats = sorted(set(sub_lbl))
cmap = plt.cm.get_cmap('tab20', len(cats))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, xy, name in zip(axes, [xy_pca, xy_tsne], ['PCA(2)', 't-SNE(2)']):
    for ci, c in enumerate(cats):
        idx = [i for i, l in enumerate(sub_lbl) if l == c]
        ax.scatter(xy[idx, 0], xy[idx, 1], s=70,
                   c=[cmap(ci)], label=c, edgecolors='k', linewidths=0.5)
    ax.set_title(f"Stage ⑤ — {name} of mask features (GT 색칠)")
    ax.legend(fontsize=7, loc='best', markerscale=0.7)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/stage5_tsne_pca.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 7. Stage ⑥ — Agglomerative cluster (cosine) + 오버레이 비교

K = GT category 수 로 클러스터링한 결과를 **GT category 색칠** 그림과 **예측 cluster 색칠** 그림 옆에 놓고 비교.
어디서 잘못 묶였는지 한눈에 보임.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

cats_unique = sorted(set(sub_lbl))
cat_to_int = {c: i for i, c in enumerate(cats_unique)}
true_int = np.array([cat_to_int[l] for l in sub_lbl])

K = max(2, len(cats_unique))
if len(sub_feat) < K:
    K = len(sub_feat)

try:
    clusterer = AgglomerativeClustering(n_clusters=K, metric='cosine', linkage='average')
    pred_int = clusterer.fit_predict(sub_feat)
except TypeError:
    clusterer = AgglomerativeClustering(n_clusters=K, affinity='cosine', linkage='average')
    pred_int = clusterer.fit_predict(sub_feat)

ari = adjusted_rand_score(true_int, pred_int)
nmi = normalized_mutual_info_score(true_int, pred_int)
purity = cluster_purity(true_int, pred_int)
h_miou, _ = hungarian_mask_miou(
    labeled_mask_ids, pred_int, sub_lbl, fake_segmap, gt_masks)
print(f"ARI={ari:.3f}  NMI={nmi:.3f}  Purity={purity:.3f}  Hung-mIoU={h_miou:.3f}")

# 오버레이용 색상
gt_cmap = plt.cm.get_cmap('tab20', len(cats_unique))
pred_cmap = plt.cm.get_cmap('tab20', K)

def colored_overlay(image_rgb, mask_ids, colors, alpha=0.6):
    out = image_rgb.copy().astype(np.float32) / 255.0
    for mid, color in zip(mask_ids, colors):
        seg = all_masks[mid]['segmentation']
        out[seg] = (1 - alpha) * out[seg] + alpha * np.array(color)[:3]
    return np.clip(out, 0, 1)

gt_colors = [gt_cmap(true_int[i]) for i in range(len(labeled_mask_ids))]
pred_colors = [pred_cmap(pred_int[i]) for i in range(len(labeled_mask_ids))]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(colored_overlay(image_rgb, labeled_mask_ids, gt_colors))
axes[0].set_title(f"GT category 색칠  (정답)")
axes[0].axis('off')
axes[1].imshow(colored_overlay(image_rgb, labeled_mask_ids, pred_colors))
axes[1].set_title(f"예측 cluster 색칠  (ARI={ari:.3f})")
axes[1].axis('off')
plt.suptitle(f"Stage ⑥ — 같은 mask 를 두 기준으로 색칠. 색이 다르면 잘못 묶임", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/stage6_cluster_overlay.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 8. Stage ⑦ — GT × Cluster confusion matrix

각 GT category 가 어떤 cluster 에 흩뿌려졌나, 한 cluster 가 여러 GT 를 묶었나.
**한 cluster 에 여러 GT 가 들어있는 행렬** = ConvNeXt-L 의 뭉침 실패.


In [ ]:
import pandas as pd
conf = np.zeros((len(cats_unique), K), dtype=np.int32)
for t, p in zip(true_int, pred_int):
    conf[t, p] += 1

df_conf = pd.DataFrame(conf, index=cats_unique,
                       columns=[f'C{c}' for c in range(K)])
print("Confusion matrix (rows=GT category, cols=predicted cluster):")
print(df_conf)

fig, ax = plt.subplots(figsize=(0.8 * K + 4, 0.4 * len(cats_unique) + 2))
im = ax.imshow(conf, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(K)); ax.set_xticklabels([f'C{c}' for c in range(K)], fontsize=9)
ax.set_yticks(range(len(cats_unique))); ax.set_yticklabels(cats_unique, fontsize=9)
ax.set_xlabel('Predicted cluster'); ax.set_ylabel('GT category')
for i in range(len(cats_unique)):
    for j in range(K):
        if conf[i, j] > 0:
            ax.text(j, i, str(conf[i, j]), ha='center', va='center',
                    color='white' if conf[i, j] > conf.max()*0.5 else 'black',
                    fontsize=9)
plt.colorbar(im, ax=ax, label='# of masks')
ax.set_title(f'Stage ⑦ — Confusion matrix\n'
             f'한 col 에 여러 GT row 가 있으면 그 cluster 가 여러 카테고리를 묶은 것 (실패)')
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/stage7_confusion.png", dpi=120, bbox_inches='tight')
plt.show()

# 자동 진단: 가장 심하게 뭉친 cluster 를 찾기
merged_clusters = []
for c in range(K):
    gts_in_c = [(cats_unique[t], int(conf[t, c])) for t in range(len(cats_unique))
                if conf[t, c] > 0]
    if len(gts_in_c) > 1:
        merged_clusters.append((c, gts_in_c))
print(f"\n=== 진단 ===")
if merged_clusters:
    print(f"여러 GT 를 한 cluster 로 뭉친 cluster: {len(merged_clusters)} 개")
    for c, gts in merged_clusters:
        items = ", ".join(f"{g}×{n}" for g, n in gts)
        print(f"  C{c}: {items}")
else:
    print("뭉친 cluster 없음 — 분리 잘 됨")


---
## Cell 9. 종합 진단 요약

각 단계 결과를 모아서 어디서 가장 큰 손해가 났는지 결론.


In [ ]:
print("=" * 70)
print(f"D arm Pipeline Diagnostic — frame={SAMPLE_FRAME}")
print("=" * 70)
print(f"\n① FastSAM 4-conf:")
for n, c in zip(LEVEL_NAMES, CONF_LEVELS):
    print(f"   conf={c} ({n:7s}) → {n_before[n]:3d} masks")
print(f"\n② NMS:")
print(f"   {total_before} → {total_after} masks  (제거 {total_before - total_after})")

n_labeled = len([l for l in gt_labels if l != 'background'])
print(f"\n③ GT-label 부여:")
print(f"   labeled mask: {n_labeled} / {total_after}  "
      f"({n_labeled/max(total_after,1)*100:.1f}%)")
print(f"   mean best-IoU: {np.mean([iou for iou in best_ious if iou > 0]):.3f}")

print(f"\n④ Feature cosine 분포:")
print(f"   same-cat pair 평균 = {np.mean(same_pairs):.3f}")
print(f"   diff-cat pair 평균 = {np.mean(diff_pairs):.3f}")
margin = np.mean(same_pairs) - np.mean(diff_pairs)
print(f"   margin             = {margin:+.3f}")
if margin < 0.05:
    print(f"   → ★ ConvNeXt-L 이 카테고리 간 의미 분리를 거의 못함 (collapse)")
elif margin < 0.15:
    print(f"   → ConvNeXt-L 의 분리력이 약함 (가능한 원인)")

print(f"\n⑥ Clustering:")
print(f"   ARI={ari:.3f}  NMI={nmi:.3f}  Purity={purity:.3f}  Hung-mIoU={h_miou:.3f}")
print(f"   GT 카테고리 {len(cats_unique)}개 → cluster {K}개")
print(f"   여러 GT 묶은 cluster: {len(merged_clusters)}개")

print(f"\n결론:")
if margin < 0.10 or len(merged_clusters) > len(cats_unique) * 0.4:
    print(f"  ★ Stage ④ (ConvNeXt-L 인코딩) 에서 의미 collapse 발생")
    print(f"     → 다른 카테고리 mask 들이 비슷한 768-D 위치에 매핑됨")
    print(f"     → 결과적으로 Stage ⑥ 에서 한 cluster 로 뭉침")
elif n_labeled / max(total_after, 1) < 0.3:
    print(f"  ★ Stage ②/③ (mask 부족) 에서 손해")
    print(f"     → FastSAM mask 가 GT 와 안 맞아 라벨 부여 비율 낮음")
else:
    print(f"  ★ 명확한 단일 실패 단계 없음 — 여러 단계 부분 손해 누적")
print("\n저장 위치:", OUT_ROOT)
